In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [3]:
sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

sub.to_csv('submission.csv', index=False)

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [7]:
print(train['answer'].value_counts().max() +
      train['answer'].value_counts().min())

814


In [8]:
import string

translator = str.maketrans('', '', string.punctuation)

cleaned_prompts = (
    train['prompt']
    .str.lower()
    .str.translate(translator)
)

vocab = set()

for text in cleaned_prompts:
    vocab.update(text.split())

print("Vocabulary size:", len(vocab))

Vocabulary size: 859


In [9]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

prompt = train.loc[train['id'] == 1, 'prompt'].iloc[0]

translator = str.maketrans('', '', string.punctuation)
clean_prompt = prompt.lower().translate(translator)

words = clean_prompt.split()
filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

print("Words left:", len(filtered_words))
print(filtered_words)

Words left: 13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = (
    train['prompt'] + ' ' +
    train['A'] + ' ' +
    train['B'] + ' ' +
    train['C'] + ' ' +
    train['D'] + ' ' +
    train['E']
)

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(combined_text)

print("Feature columns:", X.shape[1])

print("Vocabulary size:", len(vectorizer.vocabulary_))

Feature columns: 2762
Vocabulary size: 2762


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

combined_text = (
    train['prompt'] + ' ' +
    train['A'] + ' ' +
    train['B'] + ' ' +
    train['C'] + ' ' +
    train['D'] + ' ' +
    train['E']
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

row = train.loc[train['id'] == 1].iloc[0]

prompt_text = row['prompt']
option_a_text = row['A']

prompt_vec = vectorizer.transform([prompt_text])
option_a_vec = vectorizer.transform([option_a_text])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0, 0]

print(round(similarity, 4))

0.272


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

combined_text = (
    train['prompt'] + ' ' +
    train['A'] + ' ' +
    train['B'] + ' ' +
    train['C'] + ' ' +
    train['D'] + ' ' +
    train['E']
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

options = ['A', 'B', 'C', 'D', 'E']

correct = 0

for _, row in train.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])

    similarities = []

    for option in options:
        option_vec = vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0, 0]
        similarities.append(sim)

    predicted_answer = options[similarities.index(max(similarities))]

    if predicted_answer == row['answer']:
        correct += 1

accuracy = (correct / len(train)) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 13.55%


In [13]:
import numpy as np

freq = train['answer'].value_counts()

top3 = freq.index[:3].tolist()

print("Top 3:", top3)

scores = []

for actual in train['answer']:
    if actual in top3:
        scores.append(1 / (top3.index(actual) + 1))
    else:
        scores.append(0.0)

map3 = np.mean(scores)

print("MAP@3 =", map3)

Top 3: ['B', 'C', 'A']
MAP@3 = 0.42125


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

combined = (
    train['prompt'] + ' ' +
    train['A'] + ' ' +
    train['B'] + ' ' +
    train['C'] + ' ' +
    train['D'] + ' ' +
    train['E']
)

vec = TfidfVectorizer(stop_words='english')
vec.fit(combined)

options = ['A','B','C','D','E']
scores = []

for _, row in train.iterrows():
    p = vec.transform([row['prompt']])

    sims = {
        opt: cosine_similarity(
            p,
            vec.transform([row[opt]])
        )[0,0]
        for opt in options
    }

    top3 = sorted(sims, key=sims.get, reverse=True)[:3]

    scores.append(
        1/(top3.index(row['answer'])+1)
        if row['answer'] in top3
        else 0
    )

print(np.mean(scores))

0.2961666666666667
